In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-03-29 00:46:48--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-03-29 00:46:48 (31.6 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 63.0 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import gensim.downloader as api
from gensim.models import KeyedVectors

# =========================
# 1. LOAD DATA
# =========================
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

# =========================
# 2. TOKENIZATION (WORD-LEVEL)
# =========================
words = text.split()
vocab = list(set(words))
vocab_size = len(vocab)

stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}

def encode(s):
    return [stoi[w] for w in s.split() if w in stoi]

def decode(l):
    return ' '.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

# =========================
# 3. TRAIN/VAL SPLIT
# =========================
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# =========================
# 4. BATCH FUNCTION
# =========================
block_size = 32   # giảm vì word-level nặng hơn
batch_size = 32

def get_batch(split="train"):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

# =========================
# 5. LOAD WORD2VEC
# =========================
print("Loading Word2Vec...")

w2v = api.load("word2vec-google-news-300")

embedding_dim = 300

# tạo embedding matrix
embedding_matrix = torch.randn(vocab_size, embedding_dim)

for word, i in stoi.items():
    if word in w2v:
        embedding_matrix[i] = torch.tensor(w2v[word])

# =========================
# 6. MODEL
# =========================
class TinyGPT(nn.Module):
    def __init__(self, embedding_matrix):
        super().__init__()

        self.embed = nn.Embedding.from_pretrained(
            embedding_matrix,
            freeze=False  # cho phép fine-tune
        )

        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x, targets=None):
        x = self.embed(x)
        logits = self.linear(x)

        if targets is None:
            return logits, None

        B, T, C = logits.shape
        loss = F.cross_entropy(
            logits.view(B*T, C),
            targets.view(B*T)
        )
        return logits, loss

# =========================
# 7. TRAIN
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"

model = TinyGPT(embedding_matrix).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

max_steps = 2000

for step in range(max_steps):
    xb, yb = get_batch("train")
    xb, yb = xb.to(device), yb.to(device)

    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"step {step}, loss {loss.item():.4f}")

# =========================
# 8. GENERATE
# =========================
def generate(model, start_str, max_new_tokens=50):
    model.eval()

    context = torch.tensor(
        encode(start_str),
        dtype=torch.long
    ).unsqueeze(0).to(device)

    for _ in range(max_new_tokens):
        logits, _ = model(context)

        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)

        next_token = torch.multinomial(probs, num_samples=1)
        context = torch.cat([context, next_token], dim=1)

    return decode(context[0].tolist())

# =========================
# 9. TEST
# =========================
print("\n=== GENERATED TEXT ===\n")
print(generate(model, "ROMEO:", 50))
print("\n----------------------\n")
print(generate(model, "JULIET:", 50))

Loading Word2Vec...
[==================================================] 100.0% 1662.8/1662.8MB downloaded
step 0, loss 10.2055
step 100, loss 10.1159
step 200, loss 9.9510
step 300, loss 9.8752
step 400, loss 9.7898
step 500, loss 9.6853
step 600, loss 9.5894
step 700, loss 9.4715
step 800, loss 9.3897
step 900, loss 9.2721
step 1000, loss 9.1365
step 1100, loss 8.9423
step 1200, loss 8.9196
step 1300, loss 8.7223
step 1400, loss 8.6051
step 1500, loss 8.5158
step 1600, loss 8.4337
step 1700, loss 8.2259
step 1800, loss 8.2257
step 1900, loss 8.1347

=== GENERATED TEXT ===

ROMEO: begins they world. wished question,--that run? of mine: renew plainly, engaged violent officers. Herbert, Antiates, angelical! damnable live? sound'-- rids waked, child. once; transform'd pricking, commons: carry? forswearing commons' severe. bestir, Erpingham, multiplying effects, Airy pretend Mamillius: bow-boy's whereupon he; treason! descent convenience. tongue-tied: gallops Takes beguiled, frosts conies